# Cleaner

While our preprocessing has removed obvious anomalies (e.g. useless/unknown characters), some data can't really be used in our ML Model and EDA. 

Importing necessary libraries

In [34]:
import pandas as pd
from pathlib import Path

Extracting our preprocessed datasets 

In [35]:
path_prefix = '../p03_preprocessing/data/'
labels = pd.read_csv(f'{path_prefix}01_binding_labels.csv', index_col=0)
naive_features = pd.read_csv(f'{path_prefix}02_naive_processed_features.csv', index_col=0)
conjoint_triads = pd.read_csv(f'{path_prefix}03_conjoint_triads_processed_features.csv', index_col=0)
global_features = pd.read_csv(f'{path_prefix}04_antibody_only_physicochemical_properties.csv', index_col=0)

## 01 - Labels Subdataset

Checking the preprocessed `labels` dataset and the values

In [41]:
labels.head()

,is_binding_SARS-CoV2_WT,is_neutral_SARS-CoV2_WT,is_nanobody,name
0,1,1,0,Curtis_3548_S-2
1,1,0,0,Curtis_3548_S-7
2,1,0,0,Curtis_3548_RBD-15
3,1,1,0,8-D9
4,1,1,0,Sun_1G11


In [42]:
binding_counts = labels['is_binding_SARS-CoV2_WT'].value_counts()
neutral_counts = labels['is_neutral_SARS-CoV2_WT'].value_counts()
nano_counts = labels['is_nanobody'].value_counts()

summary_df = pd.concat([binding_counts, neutral_counts, nano_counts], axis=1)

summary_df.columns = ['Binding', 'Neutralization', 'Is Nanobody']
summary_df.index.name = 'Class Value'
summary_df = summary_df.fillna(0).astype(int) # Fill NaNs (like where nanobody only has 0/1)

print("Dataset Class Balance Summary:")
print(summary_df)

Dataset Class Balance Summary:
             Binding  Neutralization  Is Nanobody
Class Value                                      
1              11267            5563         1218
0               5629            8059        16700
2               1022            4296            0


Notice how we have class value of `2` which means unknown. We **cannot** use unknown features in our dataset for both EDA and especially the ML model. Hence, we remove all instances where the binding class and neutral classes' value is equal to `2`

In [43]:
labels_cleaned = labels[(labels['is_binding_SARS-CoV2_WT'] != 2) & 
                (labels['is_neutral_SARS-CoV2_WT'] != 2)]
labels_cleaned.shape

(12746, 4)

To check that they are successfully removed:

In [44]:
binding_counts = labels_cleaned['is_binding_SARS-CoV2_WT'].value_counts()
neutral_counts = labels_cleaned['is_neutral_SARS-CoV2_WT'].value_counts()
nano_counts = labels_cleaned['is_nanobody'].value_counts()

summary_df = pd.concat([binding_counts, neutral_counts, nano_counts], axis=1)

summary_df.columns = ['Binding', 'Neutralization', 'Is Nanobody']
summary_df.index.name = 'Class Value'
summary_df = summary_df.fillna(0).astype(int) # Fill NaNs (like where nanobody only has 0/1)

print("Dataset Class Balance Summary:")
print(summary_df)

Dataset Class Balance Summary:
             Binding  Neutralization  Is Nanobody
Class Value                                      
1               7494            5420          805
0               5252            7326        11941


As shown, we have successfully removed all instances with a value of `2` in the binding class or neutral class.

This cleaned dataset will then be saved in the `/cleaned_data` folder

In [45]:
output_path = Path("cleaned_data/01_binding_labels._cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
labels_cleaned.to_csv(output_path)
print(f"Saved cleaned labels to {output_path}")

Saved cleaned labels to cleaned_data/01_binding_labels._cleaned.csv


## 02 - Naive Processed Features